**Author:** Salvador Navas  
**Date:** 2025-06-27

### ESGF — Earth System Grid Federation

**ESGF** is a globally distributed data infrastructure for large-scale climate data.

- Hosts **CMIP6, CMIP5, CORDEX** datasets
- Supports **OPeNDAP** (lazy remote access) and **HTTPServer** (full download)
- Multiple worldwide nodes for redundancy

**Key portals:**
- LLNL: https://esgf-node.llnl.gov/search/cmip6/
- DKRZ: https://esgf-data.dkrz.de/search/cmip6-dkrz/
- CEDA: https://esgf-index1.ceda.ac.uk/search/cmip6-ceda/

> This notebook uses **synthetic demo data** that reproduces the CMIP6 format
> without requiring ESGF credentials or network access.
> Replace the synthetic section with actual ESGF download workflow when credentials are available.

In [1]:
# ESGF functions require network access to ESGF nodes
# Interactive map requires ipyleaflet — both skipped
from pyhydra.data_sources.climate_change import (
    get_dataset_metadata,
    get_all_urls,
    get_combination_if_complete,
    download_file,
    process_file,
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr
import tempfile, os
print('Imports OK')

Imports OK


---
## ESGF workflow — real usage

```python
# 1. Search for available models
filters = {
    'project': 'CMIP6',
    'table_id': 'day',
    'experiment_id': ['historical', 'ssp245', 'ssp585'],
    'variable_id': ['pr', 'tasmax', 'tasmin'],
}
df_meta = get_dataset_metadata(filters, limit=3000)

# 2. Check for complete variable sets
combo = get_combination_if_complete(
    model='MIROC6', experiment='historical', variant='r1i1p1f1',
    variables=['pr', 'tasmax', 'tasmin']
)

# 3. Get download URLs
urls_info = get_all_urls(combo[0]['dataset_id'])

# 4. Process and download subset
for item in urls_info:
    process_file(item['url'], 'esgf_output', lat_min=35, lat_max=44,
                 lon_min=-10, lon_max=5, url_type=item['url_type'])
```

In [2]:
# === SYNTHETIC DEMO DATA ===
# Mimics CMIP6 daily NetCDF output from ESGF (pr, tasmax, tasmin)

TMPDIR = tempfile.mkdtemp()
rng = np.random.default_rng(42)

times = pd.date_range('2000-01-01', '2014-12-31', freq='D')
lats  = np.arange(44.0, 35.0, -1.0)   # 1° grid
lons  = np.arange(-9.0, 5.0,  1.0)

doy = times.dayofyear.values[:, None, None]

# pr (kg m-2 s-1) — seasonal pattern
seasonal_pr = 3e-5 * (1 + 0.8 * np.sin(2 * np.pi * (doy - 280) / 365))
pr = (seasonal_pr * rng.gamma(0.8, 1, (len(times), len(lats), len(lons)))).astype('float32')

# tasmax (K)
seasonal_t = 298 + 15 * np.sin(2 * np.pi * (doy - 15) / 365)
tasmax = (seasonal_t + rng.normal(0, 1.5, (len(times), len(lats), len(lons)))).astype('float32')
tasmin = (tasmax - rng.uniform(6, 16, (len(times), len(lats), len(lons)))).astype('float32')

ds_daily = xr.Dataset(
    {
        'pr':     (['time', 'lat', 'lon'], pr),
        'tasmax': (['time', 'lat', 'lon'], tasmax),
        'tasmin': (['time', 'lat', 'lon'], tasmin),
    },
    coords={'time': times, 'lat': lats, 'lon': lons},
    attrs={
        'source_id': 'DEMO-MIROC6',
        'experiment_id': 'historical',
        'variant_label': 'r1i1p1f1',
        'institution': 'DEMO',
    }
)
for var, unit in [('pr', 'kg m-2 s-1'), ('tasmax', 'K'), ('tasmin', 'K')]:
    ds_daily[var].attrs['units'] = unit

NC_FILE = os.path.join(TMPDIR, 'DEMO-MIROC6_historical_pr_tasmax_tasmin.nc')
ds_daily.to_netcdf(NC_FILE)
print(f'Synthetic ESGF CMIP6 NetCDF created: {NC_FILE}')
print(ds_daily)

Synthetic ESGF CMIP6 NetCDF created: /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/tmpsu0rwshr/DEMO-MIROC6_historical_pr_tasmax_tasmin.nc
<xarray.Dataset> Size: 8MB
Dimensions:  (time: 5479, lat: 9, lon: 14)
Coordinates:
  * time     (time) datetime64[ns] 44kB 2000-01-01 2000-01-02 ... 2014-12-31
  * lat      (lat) float64 72B 44.0 43.0 42.0 41.0 40.0 39.0 38.0 37.0 36.0
  * lon      (lon) float64 112B -9.0 -8.0 -7.0 -6.0 -5.0 ... 0.0 1.0 2.0 3.0 4.0
Data variables:
    pr       (time, lat, lon) float32 3MB 6.848e-05 9.597e-05 ... 2.36e-06
    tasmax   (time, lat, lon) float32 3MB 294.8 291.6 296.1 ... 293.2 293.2
    tasmin   (time, lat, lon) float32 3MB 281.4 284.8 289.9 ... 285.4 278.3
Attributes:
    source_id:      DEMO-MIROC6
    experiment_id:  historical
    variant_label:  r1i1p1f1
    institution:    DEMO


---
## 1. Mimic metadata DataFrame

In [3]:
# Simulate the DataFrame returned by get_dataset_metadata()
df_meta_demo = pd.DataFrame([
    {'dataset_id': 'CMIP6.day.DEMO.MIROC6.historical.r1i1p1f1.day.pr',
     'model': 'MIROC6', 'experiment': 'historical', 'variable': 'pr',
     'variant': 'r1i1p1f1', 'table': 'day',
     'start': '2000-01-01', 'end': '2014-12-31'},
    {'dataset_id': 'CMIP6.day.DEMO.MIROC6.historical.r1i1p1f1.day.tasmax',
     'model': 'MIROC6', 'experiment': 'historical', 'variable': 'tasmax',
     'variant': 'r1i1p1f1', 'table': 'day',
     'start': '2000-01-01', 'end': '2014-12-31'},
    {'dataset_id': 'CMIP6.day.DEMO.MIROC6.historical.r1i1p1f1.day.tasmin',
     'model': 'MIROC6', 'experiment': 'historical', 'variable': 'tasmin',
     'variant': 'r1i1p1f1', 'table': 'day',
     'start': '2000-01-01', 'end': '2014-12-31'},
])
print('Simulated metadata DataFrame (from get_dataset_metadata):')
df_meta_demo

Simulated metadata DataFrame (from get_dataset_metadata):


,dataset_id,model,experiment,variable,variant,table,start,end
0,CMIP6.day.DEMO.MIROC6.historical.r1i1p1f1.day.pr,MIROC6,historical,pr,r1i1p1f1,day,2000-01-01,2014-12-31
1,CMIP6.day.DEMO.MIROC6.historical.r1i1p1f1.day....,MIROC6,historical,tasmax,r1i1p1f1,day,2000-01-01,2014-12-31
2,CMIP6.day.DEMO.MIROC6.historical.r1i1p1f1.day....,MIROC6,historical,tasmin,r1i1p1f1,day,2000-01-01,2014-12-31


---
## 2. Extract and analyse point time series

In [4]:
LAT, LON = 40.0, -4.0   # Madrid

ds = xr.open_dataset(NC_FILE)
pt = ds.sel(lat=LAT, lon=LON, method='nearest')
df_pt = pt.to_dataframe().reset_index()[['time', 'pr', 'tasmax', 'tasmin']]
df_pt['pr_mm_day']  = df_pt['pr'] * 86400
df_pt['tasmax_c']   = df_pt['tasmax'] - 273.15
df_pt['tasmin_c']   = df_pt['tasmin'] - 273.15
df_pt['tmean_c']    = (df_pt['tasmax_c'] + df_pt['tasmin_c']) / 2
df_pt = df_pt.set_index('time').sort_index()
print(df_pt[['pr_mm_day', 'tasmax_c', 'tasmin_c']].describe())
df_pt.head()

         pr_mm_day     tasmax_c     tasmin_c
count  5479.000000  5479.000000  5479.000000
mean      2.033036    24.849665    13.822254
std       2.883784    10.708743    11.118884
min       0.000035     5.698853   -10.019562
25%       0.312359    14.382782     3.789124
50%       0.960836    24.702423    13.661865
75%       2.571098    35.294296    23.872574
max      27.166904    45.552185    36.713867


,pr,tasmax,tasmin,pr_mm_day,tasmax_c,tasmin_c,tmean_c
time,,,,,,,
2000-01-01,0.000001,293.282715,280.492371,0.087533,20.132721,7.342377,13.737549
2000-01-02,0.000004,293.447327,281.925446,0.335731,20.297333,8.775452,14.536392
2000-01-03,0.000128,296.873413,285.575684,11.063967,23.723419,12.425690,18.074554
2000-01-04,0.000110,294.551331,285.055725,9.540928,21.401337,11.905731,16.653534
2000-01-05,0.000025,296.562958,285.667145,2.152693,23.412964,12.517151,17.965057


---
## 3. Visualisation

In [5]:
annual_pr   = df_pt['pr_mm_day'].resample('YE').sum()
annual_tmax = df_pt['tasmax_c'].resample('YE').mean()
monthly_pr  = df_pt['pr_mm_day'].groupby(df_pt.index.month).mean()

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].bar(annual_pr.index.year, annual_pr.values, color='steelblue', alpha=0.8)
axes[0].set_ylabel('Annual precipitation (mm)')
axes[0].set_title(
    'ESGF CMIP6 — MIROC6 historical — Madrid (synthetic demo)', fontsize=13
)
axes[0].grid(alpha=0.3, axis='y')

axes[1].plot(annual_tmax.index.year, annual_tmax.values, 'o-', color='tomato', lw=1.5)
axes[1].set_ylabel('Annual mean Tmax (°C)')
axes[1].set_title('Annual mean daily maximum temperature')
axes[1].grid(alpha=0.3)

axes[2].bar(monthly_pr.index, monthly_pr.values, color='royalblue', alpha=0.85)
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'])
axes[2].set_ylabel('Mean daily precipitation (mm)')
axes[2].set_title('Seasonal Precipitation Regime')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/ESGF_download.png', dpi=100, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/ESGF_download.png')
ds.close()

Plot saved to /tmp/ESGF_download.png
